# 2단계 피처 엔지니어링

1단계에서 생성한 도시별 샘플 CSV를 Google Drive에서 불러오고, 텍스트/날짜/유저/식당/상권 클러스터 파생변수를 추가한 뒤 `*_features.csv` 파일로 저장합니다.

이 노트북은 Colab 실행을 기준으로 작성되었습니다.

## 1. 환경 설정 및 경로 지정

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project/Team-6')

def find_input_dir(project_root):
    candidates = [
        project_root / 'data' / 'interim',
        project_root / 'sampling',
        project_root / 'eda_sampling',
    ]
    for candidate in candidates:
        if (candidate / 'yelp_subset_philly_15k.csv').exists():
            print(f'Detected INPUT_DIR: {candidate}')
            return candidate

    matches = list(project_root.rglob('yelp_subset_philly_15k.csv'))
    if matches:
        detected_dir = matches[0].parent
        print(f'Detected INPUT_DIR: {detected_dir}')
        return detected_dir

    raise FileNotFoundError(f'yelp_subset_philly_15k.csv was not found under {project_root}')


INPUT_DIR = find_input_dir(PROJECT_ROOT)
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ml_project')
BUSINESS_LOCATION_CSV_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'interim' / 'business_location.csv',
    PROJECT_ROOT / 'feature_engineering' / 'business_location.csv',
    PROJECT_ROOT / 'data' / 'business_location.csv',
    PROJECT_ROOT / 'eda_sampling' / 'business_location.csv',
]

BUSINESS_JSON_CANDIDATES = [
    Path('/content/yelp_data/yelp_academic_dataset_business.json'),
    PROJECT_ROOT / 'data' / 'yelp_academic_dataset_business.json',
    PROJECT_ROOT / 'sampling' / 'yelp_academic_dataset_business.json',
    PROJECT_ROOT / 'eda_sampling' / 'yelp_academic_dataset_business.json',
]

CITY_CONFIGS = [
    {
        'city': 'Philadelphia',
        'input_file': 'yelp_subset_philly_15k.csv',
        'output_file': 'yelp_subset_philly_15k_features.csv',
    },
    {
        'city': 'Tucson',
        'input_file': 'yelp_subset_tucson_15k.csv',
        'output_file': 'yelp_subset_tucson_15k_features.csv',
    },
    {
        'city': 'New Orleans',
        'input_file': 'yelp_subset_new_orleans_15k.csv',
        'output_file': 'yelp_subset_new_orleans_15k_features.csv',
    },
]

KMEANS_CLUSTERS = 5
RANDOM_STATE = 42

print(f'INPUT_DIR: {INPUT_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
Detected INPUT_DIR: /content/drive/MyDrive/ml_project/Team-6/data/interim
INPUT_DIR: /content/drive/MyDrive/ml_project/Team-6/data/interim
OUTPUT_DIR: /content/drive/MyDrive/ml_project/Team-6/data/processed


## 2. 입력 파일 확인

In [2]:
def check_input_files(city_configs=CITY_CONFIGS):
    missing_files = []
    for config in city_configs:
        input_path = INPUT_DIR / config['input_file']
        if not input_path.exists():
            missing_files.append(input_path)

    if missing_files:
        missing_text = '\n'.join(str(path) for path in missing_files)
        raise FileNotFoundError(f'Input CSV files were not found:\n{missing_text}')

    print('All input CSV files exist.')


check_input_files()

All input CSV files exist.


## 3. 공통 함수 정의

In [3]:
def add_text_features(df):
    df = df.copy()
    df['text'] = df['text'].fillna('').astype(str)

    df['text_length'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()
    df['sentence_count'] = df['text'].str.count(r'[.!?]+').clip(lower=1)
    df['avg_word_length'] = df['text_length'] / df['word_count'].clip(lower=1)
    df['uppercase_ratio'] = df['text'].apply(
        lambda text: sum(1 for char in text if char.isupper()) / max(len(text), 1)
    )
    df['exclamation_count'] = df['text'].str.count('!')
    df['question_count'] = df['text'].str.count(r'\?')
    df['engagement_sum'] = df[['useful', 'funny', 'cool']].fillna(0).sum(axis=1)
    return df


def add_date_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    df['review_year'] = df['date'].dt.year
    df['review_month'] = df['date'].dt.month
    df['review_dayofweek'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['review_dayofweek'].isin([5, 6]).astype(int)
    return df


def add_user_features(df):
    df = df.copy()
    user_stats = df.groupby('user_id')['stars'].agg(
        user_review_count='count',
        user_avg_stars='mean'
    ).reset_index()

    df = df.merge(user_stats, on='user_id', how='left')
    df['user_star_deviation'] = df['stars'] - df['user_avg_stars']
    return df


def add_business_features(df):
    df = df.copy()
    business_stats = df.groupby('business_id')['stars'].agg(
        business_review_count='count',
        business_avg_stars='mean'
    ).reset_index()

    df = df.merge(business_stats, on='business_id', how='left')
    df['business_star_deviation'] = df['stars'] - df['business_avg_stars']
    return df


def find_business_location_csv_path():
    for path in BUSINESS_LOCATION_CSV_CANDIDATES:
        if path.exists():
            print(f'Detected business location CSV: {path}')
            return path

    matches = list(DRIVE_PROJECT_ROOT.rglob('business_location.csv'))
    if matches:
        detected_path = matches[0]
        print(f'Detected business location CSV: {detected_path}')
        return detected_path

    return None


def find_business_json_path():
    for path in BUSINESS_JSON_CANDIDATES:
        if path.exists():
            print(f'Detected business JSON: {path}')
            return path

    matches = list(DRIVE_PROJECT_ROOT.rglob('yelp_academic_dataset_business.json'))
    if matches:
        detected_path = matches[0]
        print(f'Detected business JSON: {detected_path}')
        return detected_path

    return None


def load_business_locations(business_ids):
    location_csv_path = find_business_location_csv_path()
    if location_csv_path is not None:
        location_df = pd.read_csv(location_csv_path)
        return location_df.loc[
            location_df['business_id'].isin(business_ids),
            ['business_id', 'latitude', 'longitude']
        ].drop_duplicates('business_id')

    business_json_path = find_business_json_path()
    if business_json_path is None:
        print('business location CSV or business.json was not found. Location and K-Means features will be skipped.')
        print('Expected filenames: business_location.csv or yelp_academic_dataset_business.json')
        print(f'Searched under: {DRIVE_PROJECT_ROOT}')
        return None

    print(f'Loading business locations from: {business_json_path}')
    biz_df = pd.read_json(business_json_path, lines=True)
    location_df = biz_df.loc[
        biz_df['business_id'].isin(business_ids),
        ['business_id', 'latitude', 'longitude']
    ].drop_duplicates('business_id')
    return location_df


def add_location_features(df, location_df):
    df = df.copy()
    if location_df is None:
        df['latitude'] = np.nan
        df['longitude'] = np.nan
        return df

    return df.merge(location_df, on='business_id', how='left')


def add_kmeans_cluster(df, n_clusters=KMEANS_CLUSTERS, random_state=RANDOM_STATE):
    df = df.copy()
    df['business_cluster'] = np.nan

    coords = df[['latitude', 'longitude']].dropna()
    if coords.empty:
        print('No valid coordinates. K-Means clustering skipped.')
        return df

    unique_count = coords.drop_duplicates().shape[0]
    cluster_count = min(n_clusters, unique_count)
    if cluster_count < 2:
        print('Not enough unique coordinates. K-Means clustering skipped.')
        return df

    scaler = StandardScaler()
    coords_scaled = scaler.fit_transform(coords)

    kmeans = KMeans(n_clusters=cluster_count, random_state=random_state, n_init='auto')
    df.loc[coords.index, 'business_cluster'] = kmeans.fit_predict(coords_scaled)
    df['business_cluster'] = df['business_cluster'].astype('Int64')
    return df


def add_all_features(df, location_df):
    df = add_text_features(df)
    df = add_date_features(df)
    df = add_user_features(df)
    df = add_business_features(df)
    df = add_location_features(df, location_df)
    df = add_kmeans_cluster(df)
    return df


def summarize_features(df):
    print('Shape:', df.shape)
    print('\nTarget distribution:')
    print(df['is_positive'].value_counts(dropna=False))
    print('\nTop missing columns:')
    print(df.isnull().sum().sort_values(ascending=False).head(15))

## 4. 위치 정보 로드

`business.json`이 `/content/yelp_data` 또는 Drive의 `Team-6/data`에 있으면 위도/경도를 병합하고 K-Means 클러스터를 생성합니다. 파일이 없으면 위치 기반 피처만 건너뜁니다.

In [4]:
all_business_ids = set()
for config in CITY_CONFIGS:
    temp_df = pd.read_csv(INPUT_DIR / config['input_file'], usecols=['business_id'])
    all_business_ids.update(temp_df['business_id'].dropna().unique())

location_df = load_business_locations(all_business_ids)
if location_df is not None:
    print('Location rows:', location_df.shape[0])
    display(location_df.head())

Detected business location CSV: /content/drive/MyDrive/ml_project/Team-6/data/interim/business_location.csv
Location rows: 7012


,business_id,latitude,longitude
3,MTSW4McQd7CbVtyjqoe9mw,39.955505,-75.155564
15,MUTTqe8uqyMdBl186RmNeA,39.953949,-75.143226
19,ROeacJQwBeh05Rqg7F6TCg,39.943223,-75.162568
28,QdN72BWoyFypdGJhhI5r7g,39.939825,-75.157447
31,Mjboz24M9NlBeiOJKLEd_Q,40.022466,-75.218314


## 5. 도시별 피처 생성 및 저장

In [5]:
feature_datasets = {}

for config in CITY_CONFIGS:
    city = config['city']
    input_path = INPUT_DIR / config['input_file']
    output_path = OUTPUT_DIR / config['output_file']

    print(f'\n[{city}] Feature engineering started')
    df = pd.read_csv(input_path)
    print('Original shape:', df.shape)

    feature_df = add_all_features(df, location_df)
    summarize_features(feature_df)

    feature_df.to_csv(output_path, index=False)
    feature_datasets[city] = feature_df
    print(f'Saved: {output_path}')

print('\nAll feature engineering completed.')


[Philadelphia] Feature engineering started
Original shape: (15000, 10)
Shape: (15000, 31)

Target distribution:
is_positive
1    11880
0     3120
Name: count, dtype: int64

Top missing columns:
review_id          0
user_id            0
business_id        0
stars              0
useful             0
funny              0
cool               0
text               0
date               0
is_positive        0
text_length        0
word_count         0
sentence_count     0
avg_word_length    0
uppercase_ratio    0
dtype: int64
Saved: /content/drive/MyDrive/ml_project/Team-6/data/processed/yelp_subset_philly_15k_features.csv

[Tucson] Feature engineering started
Original shape: (15000, 10)
Shape: (15000, 31)

Target distribution:
is_positive
1    11169
0     3831
Name: count, dtype: int64

Top missing columns:
review_id          0
user_id            0
business_id        0
stars              0
useful             0
funny              0
cool               0
text               0
date               0


## 6. 결과 확인

In [6]:
for config in CITY_CONFIGS:
    output_path = OUTPUT_DIR / config['output_file']
    print(output_path, output_path.exists())

sample_city = CITY_CONFIGS[0]['city']
display(feature_datasets[sample_city].head())

/content/drive/MyDrive/ml_project/Team-6/data/processed/yelp_subset_philly_15k_features.csv True
/content/drive/MyDrive/ml_project/Team-6/data/processed/yelp_subset_tucson_15k_features.csv True
/content/drive/MyDrive/ml_project/Team-6/data/processed/yelp_subset_new_orleans_15k_features.csv True


,review_id,user_id,business_id,stars,useful,funny,cool,text,date,is_positive,...,is_weekend,user_review_count,user_avg_stars,user_star_deviation,business_review_count,business_avg_stars,business_star_deviation,latitude,longitude,business_cluster
0,yrgcQRzGdvODnmPgGzsLPg,8Oj4m1xxNoI1P_1mek9fQQ,QWqKTWQ2OiDgo3dzNkpung,5,3,0,1,After hearing rumors of delicious Soft Tofu pl...,2010-08-18 18:20:37,1,...,0,6,3.666667,1.333333,13,4.538462,0.461538,40.052064,-75.127394,4
1,QxtMgdD8Pz0w7XAUIPi3lg,KXYRjdlgg-Gw7mwF22ZAJg,0h4Oo6W8mG4HEYQzZRkoOA,4,0,0,1,Tasty new location in a plaza full of vietname...,2017-05-29 20:31:01,1,...,0,2,4.000000,0.000000,2,4.000000,0.000000,40.034890,-75.100337,4
2,gl3C_xngo9dUwJVz0lC2sw,gOAvWocRyGbKvUaitVy8fQ,nP4LqTcK1USiw5WnbOjW7A,5,0,0,0,"i ADMIT, I COME OFTEN: because all the food h...",2010-12-11 02:27:30,1,...,1,12,4.750000,0.250000,1,5.000000,0.000000,39.956716,-75.210034,0
3,PKHKI7US_xXiNH4ml6k5GA,-xw19I8BbBP35H2sCN2uRQ,WYel6hMEEPNwzVUlVBacbw,4,1,0,1,I really like this place. The salads are excel...,2017-03-26 17:49:40,1,...,1,1,4.000000,0.000000,11,3.454545,0.545455,39.950723,-75.167501,0
4,QcbZk6Wj8Rh8cZZeA7OtVA,jugISJYY36jq2Bse8dKugg,mlrYh-m3Zlqp9CTeSvdOsQ,5,0,0,0,Teriyaki and spicy chicken bowls were bombbbbb...,2016-08-25 00:32:01,1,...,0,1,5.000000,0.000000,3,4.666667,0.333333,39.948994,-75.159035,2


## 7. Save Final Feature CSV Files

Save the final feature-engineered datasets to `data/processed/`. This cell also verifies the saved file paths, row counts, column counts, and file sizes.


In [ ]:
saved_outputs = []

for config in CITY_CONFIGS:
    city = config['city']
    output_path = OUTPUT_DIR / config['output_file']

    if city not in feature_datasets:
        raise KeyError(f"{city} data was not found in feature_datasets. Run the feature engineering cell first.")

    feature_datasets[city].to_csv(output_path, index=False)
    saved_outputs.append({
        'city': city,
        'path': str(output_path),
        'rows': len(feature_datasets[city]),
        'columns': feature_datasets[city].shape[1],
        'file_size_mb': output_path.stat().st_size / 1024 / 1024,
    })

saved_outputs_df = pd.DataFrame(saved_outputs)
display(saved_outputs_df)
print('Final feature CSV files saved successfully.')
